In [ ]:
# Install necessary packages if they are not already installed
if (!require("tidyverse")) install.packages("tidyverse")
if (!require("scales")) install.packages("scales")
# ggnewscale is essential for having separate color scales in the same plot
if (!require("ggnewscale")) install.packages("ggnewscale") 
if (!require("RColorBrewer")) install.packages("RColorBrewer")

library(tidyverse)
library(scales)
library(ggnewscale)
library(RColorBrewer)

# --- Configuration ---
inputFolder <- "dataForRScripts"
output_dir <- "OutputImages"
inputFilename <- "Statistical_3km_Spatial_Climatologies.csv"
input_filepath <- file.path(inputFolder, inputFilename)

# Create output directory if it doesn't exist
if (!dir.exists(output_dir)) {
  dir.create(output_dir)
}

# --- Data Loading and Prep ---
if (!file.exists(input_filepath)) {
  stop(paste("input file not found:", input_filepath, ". Run Python script first."))
}
data <- read_csv(input_filepath, show_col_types = FALSE)

# Convert categorical columns to factors
data <- data %>%
  mutate(Region = as.factor(Region),
         Variable = as.factor(Variable),
         Type = as.factor(Type),
         Scenario = as.factor(Scenario))

# Define plot order for scenarios (Historical must be first)
scenario_order <- c("Historical Climate", "SSP 1-2.6", "SSP 2-4.5", "SSP 3-7.0", "SSP 5-8.5")
# Filter out scenarios that might be missing in the data (e.g. if processing failed for one SSP)
valid_scenarios <- intersect(scenario_order, levels(data$Scenario))
data$Scenario <- factor(data$Scenario, levels = valid_scenarios)

# Define Quarter mapping if applicable (for Precipitation)
if ("quarter" %in% names(data)) {
  # Mapping based on QS-DEC resampling in Python (1=DJF, 2=MAM, 3=JJA, 4=SON)
  quarter_map <- c(`1` = "Q1 (DJF)", `2` = "Q2 (MAM)", `3` = "Q3 (JJA)", `4` = "Q4 (SON)")
  data$QuarterName <- quarter_map[as.character(data$quarter)]
  data$QuarterName <- factor(data$QuarterName, levels = unname(quarter_map))
}

# --- Plotting Function ---

create_spatial_comparison_plot <- function(plot_data, region_name, var_name, plot_title) {
  
  # Separate baseline and anomaly data
  baseline_data <- plot_data %>% filter(Type == "Baseline", Scenario == "Historical Climate")
  anomaly_data <- plot_data %>% filter(Type == "Anomaly", Scenario != "Historical Climate")
  
  # Determine color scales and limits
  
  # 1. Baseline (Sequential scale)
  baseline_limits <- range(baseline_data$Value, na.rm = TRUE)
  
  # Define colors and titles
  if (grepl("T_Avg", var_name)) {
      baseline_cmap_option <- "plasma" # Good sequential scale for temperature
      baseline_legend_title <- "Temp (°C)"
      anomaly_cmap_palette <- "RdBu"
      anomaly_legend_title <- "ΔTemp (°C)"
      # Direction -1 reverses RdBu so Red is positive/hot
      anomaly_direction <- -1 
  } else {
      baseline_cmap_option <- "Blues"
      baseline_legend_title <- "Precip (mm/season)"
      anomaly_cmap_palette <- "BrBG"
      anomaly_legend_title <- "ΔPrecip (%)"
      anomaly_direction <- 1
  }

  
  # 2. Anomalies (Diverging scale)
  anomaly_limits <- range(anomaly_data$Value, na.rm = TRUE)
  # Center the diverging scale around 0
  max_abs_anomaly <- max(abs(anomaly_limits), na.rm = TRUE)
  
  # Handle case where data might be missing or infinite
  if (!is.finite(max_abs_anomaly) || max_abs_anomaly == 0) {
      max_abs_anomaly <- 1 # Fallback
  }
  
  # Set sensible limits for precipitation percentage change if they are extreme
  if (grepl("Precip", var_name) && max_abs_anomaly > 100) {
      max_abs_anomaly <- 100
      print("  Capping precipitation anomaly scale at +/- 100% for better visualization.")
  }
  
  # Add a little buffer and round for cleaner legend breaks
  if (max_abs_anomaly < 5) {
       max_abs_anomaly <- ceiling(max_abs_anomaly * 1.1)
  } else {
      # Round up to nearest 5 or 10
      round_factor <- if (max_abs_anomaly < 50) 5 else 10
      max_abs_anomaly <- ceiling(max_abs_anomaly / round_factor) * round_factor
  }

  diverging_limits <- c(-max_abs_anomaly, max_abs_anomaly)

  
  # Start the plot
  p <- ggplot() +
    
    # --- Plot Baseline Data ---
    # Use geom_raster for efficient plotting of gridded data
    geom_raster(data = baseline_data, aes(x = x, y = y, fill = Value)) +
    
    # Apply baseline color scale
    # Use scale_fill_viridis_c for plasma/viridis, scale_fill_distiller for Blues
    if (baseline_cmap_option == "Blues") {
        p <- p + scale_fill_distiller(palette = baseline_cmap_option, direction = 1, limits = baseline_limits, name = baseline_legend_title)
    } else {
        p <- p + scale_fill_viridis_c(option = baseline_cmap_option, limits = baseline_limits, name = baseline_legend_title)
    }

    
    # --- Introduce New Fill Scale for Anomalies ---
    p <- p + ggnewscale::new_scale_fill() +
    
    # --- Plot Anomaly Data ---
    geom_raster(data = anomaly_data, aes(x = x, y = y, fill = Value)) +
    
    # Apply anomaly color scale (Diverging)
    # Use oob=squish to handle values outside the defined limits (especially for capped precip)
    scale_fill_distiller(palette = anomaly_cmap_palette, direction = anomaly_direction, 
                         limits = diverging_limits, name = anomaly_legend_title,
                         oob = scales::squish) + 
    
    # Faceting
    # If the data includes Quarters (like Precip), facet by Quarter (rows) and Scenario (cols)
    if ("QuarterName" %in% names(plot_data)) {
      p <- p + facet_grid(QuarterName ~ Scenario)
    } else {
      # Otherwise, just facet by Scenario (e.g., for Temperature)
      p <- p + facet_wrap(~ Scenario, nrow = 1)
    }
    
    # Styling
    labs(title = plot_title,
         subtitle = paste(region_name, "| Statistical 3km | Baseline: 1985-2014 | Future: 2071-2100")) +
    coord_fixed() + # Ensure aspect ratio is correct for spatial data
    theme_bw(base_size = 12) +
    theme(axis.title = element_blank(),
          axis.text = element_blank(), # Remove Lat/Lon labels for cleaner maps
          axis.ticks = element_blank(),
          panel.grid = element_blank(),
          plot.title = element_text(hjust = 0.5, face = "bold"),
          plot.subtitle = element_text(hjust = 0.5),
          strip.background = element_rect(fill = "grey95"),
          strip.text = element_text(face = "bold"),
          legend.position = "right",
          legend.key.height = unit(1.5, "cm"))
  
  return(p)
}

# --- Generate and Save Plots ---

regions <- levels(data$Region)
variables <- levels(data$Variable)

for (region in regions) {
  print(paste("Generating plots for region:", region))
  
  # 1. Precipitation (All Quarters)
  if ("Precip_All_Q" %in% variables) {
    var_key <- "Precip_All_Q"
    plot_data <- data %>% filter(Region == region, Variable == var_key)
    if (nrow(plot_data) > 0) {
        title <- "Quarterly Precipitation Climatology and Anomalies"
        p_precip <- create_spatial_comparison_plot(plot_data, region, var_key, title)
        
        # Save plot (requires a larger canvas due to 4 rows x 5 columns facets)
        output_filename <- file.path(output_dir, paste0(region, "_Statistical_3km_Spatial_Precip.png"))
        # Adjust height based on the aspect ratio of the region
        plot_height <- if (region == "Mojave") 14 else 12
        ggsave(output_filename, plot = p_precip, width = 15, height = plot_height, dpi = 300)
        print(paste("Saved plot to:", output_filename))
    }
  }
  
  # 2. Temperature (Warmest Quarter)
  if ("T_Avg_Warmest_Q" %in% variables) {
    var_key <- "T_Avg_Warmest_Q"
    plot_data <- data %>% filter(Region == region, Variable == var_key)
    if (nrow(plot_data) > 0) {
        title <- "Warmest Quarter Temperature Climatology and Anomalies"
        p_temp_warm <- create_spatial_comparison_plot(plot_data, region, var_key, title)
        
        # Save plot (1 row x 5 columns facets)
        output_filename <- file.path(output_dir, paste0(region, "_Statistical_3km_Spatial_Temp_WarmestQ.png"))
        # Adjust height based on typical aspect ratio of the parks
        plot_height <- if (region == "Mojave") 5.5 else 4.5
        ggsave(output_filename, plot = p_temp_warm, width = 15, height = plot_height, dpi = 300)
        print(paste("Saved plot to:", output_filename))
    }
  }

  # 3. Temperature (Coldest Quarter)
  if ("T_Avg_Coldest_Q" %in% variables) {
    var_key <- "T_Avg_Coldest_Q"
    plot_data <- data %>% filter(Region == region, Variable == var_key)
    if (nrow(plot_data) > 0) {
        title <- "Coldest Quarter Temperature Climatology and Anomalies"
        p_temp_cold <- create_spatial_comparison_plot(plot_data, region, var_key, title)
        
        # Save plot (1 row x 5 columns facets)
        output_filename <- file.path(output_dir, paste0(region, "_Statistical_3km_Spatial_Temp_ColdestQ.png"))
        plot_height <- if (region == "Mojave") 5.5 else 4.5
        ggsave(output_filename, plot = p_temp_cold, width = 15, height = plot_height, dpi = 300)
        print(paste("Saved plot to:", output_filename))
    }
  }
}

print("R script finished.")